# 02. 어린이인구 전처리

2025년 12월 어린이(0~12세) 인구 데이터를 사고 데이터의 229개 지역 기준에 맞춰 정리합니다.

**입력**
- `../data_raw/202512_202512_연령별인구현황_월간.csv`
- `../data_clean/accident_2020_2025_child_clean.csv`

**출력**
- `../data_clean/population_2025_child_clean.csv`

In [1]:
import pandas as pd

population_raw = pd.read_csv(
    "../data_raw/202512_202512_연령별인구현황_월간.csv",
    encoding="cp949"
)

accident = pd.read_csv(
    "../data_clean/accident_2020_2025_child_clean.csv",
    encoding="utf-8-sig"
)

print("인구 원본:", population_raw.shape)
print("사고 지역 수:", accident["sido_sgg_nm"].nunique())
display(population_raw.head())

인구 원본: (294, 46)
사고 지역 수: 229


,행정구역,2025년12월_계_총인구수,2025년12월_계_연령구간인구수,2025년12월_계_0세,2025년12월_계_1세,2025년12월_계_2세,2025년12월_계_3세,2025년12월_계_4세,2025년12월_계_5세,2025년12월_계_6세,...,2025년12월_여_3세,2025년12월_여_4세,2025년12월_여_5세,2025년12월_여_6세,2025년12월_여_7세,2025년12월_여_8세,2025년12월_여_9세,2025년12월_여_10세,2025년12월_여_11세,2025년12월_여_12세
0,서울특별시 (1100000000),"9,299,548","661,214","43,645","39,703","37,268","39,955","42,056","43,053","47,447",...,"19,571","20,482","20,878","22,735","24,432","26,912","30,335","32,781","32,569","32,396"
1,서울특별시 종로구 (1111000000),"137,048","7,488",427,444,416,445,472,477,526,...,217,230,229,265,285,315,333,410,404,421
2,서울특별시 중구 (1114000000),"117,805","6,585",601,484,410,511,501,462,459,...,253,273,217,216,241,246,280,260,269,259
3,서울특별시 용산구 (1117000000),"201,894","13,338","1,052",951,852,917,912,955,"1,016",...,463,430,470,494,499,531,562,577,573,570
4,서울특별시 성동구 (1120000000),"274,606","20,704","1,812","1,587","1,365","1,485","1,540","1,484","1,512",...,701,760,688,713,784,760,829,857,851,801


## 1) 필요한 컬럼 추출 및 숫자형 변환

In [2]:
population = population_raw[
    ["행정구역", "2025년12월_계_연령구간인구수"]
].copy()

population.columns = [
    "sido_sgg_nm",
    "child_population_0_12"
]

population["sido_sgg_nm"] = (
    population["sido_sgg_nm"]
    .str.replace(r"\s*\(\d+\)\s*$", "", regex=True)
    .str.strip()
)

population["child_population_0_12"] = pd.to_numeric(
    population["child_population_0_12"]
    .astype(str)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

display(population.head())

,sido_sgg_nm,child_population_0_12
0,서울특별시,661214
1,서울특별시 종로구,7488
2,서울특별시 중구,6585
3,서울특별시 용산구,13338
4,서울특별시 성동구,20704


## 2) 사고 데이터의 행정구역 명칭과 통일

In [3]:
population["sido_sgg_nm"] = (
    population["sido_sgg_nm"]
    .str.replace("^강원특별자치도", "강원도", regex=True)
    .str.replace("^전북특별자치도", "전라북도", regex=True)
    .str.replace("^제주특별자치도", "제주도", regex=True)
)

population = population.drop_duplicates(
    subset=["sido_sgg_nm"],
    keep="first"
)

accident_regions = set(accident["sido_sgg_nm"].dropna().unique())

population_final = population[
    population["sido_sgg_nm"].isin(accident_regions)
].copy()

missing = sorted(
    accident_regions - set(population_final["sido_sgg_nm"])
)

print("최종 인구 지역 수:", population_final["sido_sgg_nm"].nunique())
print("사고에는 있지만 인구에는 없는 지역:", missing)
print("중복 지역:", population_final["sido_sgg_nm"].duplicated().sum())

최종 인구 지역 수: 229
사고에는 있지만 인구에는 없는 지역: []
중복 지역: 0


In [4]:
population_final = population_final.reset_index(drop=True)

population_final.to_csv(
    "../data_clean/population_2025_child_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", population_final.shape)
display(population_final.head())

저장 완료: (229, 2)


,sido_sgg_nm,child_population_0_12
0,서울특별시 종로구,7488
1,서울특별시 중구,6585
2,서울특별시 용산구,13338
3,서울특별시 성동구,20704
4,서울특별시 광진구,19667
